In [ ]:
!pip install -U torchao

In [ ]:
import os, inspect, torch, torch.nn as nn, torch.nn.functional as F
import soundfile as sf, torchaudio
import pandas as pd
import numpy as np
from transformers import AutoConfig, AutoModel, AutoFeatureExtractor
from peft import LoraConfig, get_peft_model
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, inspect, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoConfig, AutoModel, AutoFeatureExtractor
from peft import LoraConfig, get_peft_model

MID = "google/embeddinggemma-2"
SR = 16_000                              # rate the feature extractor is told; samples go in unchanged
WIN = 5120                               # window length in samples (inputs are (B, WIN))
NUM_CLASSES = 5
PROJ_DIM = 128
RECON_W = 1.0                             # weight of the window reconstruction loss
torch.set_num_threads(os.cpu_count())

# ---- keep only the audio tower (no vision, no text) ----
cfg = AutoConfig.from_pretrained(MID)
cfg.vision_config = None
base = AutoModel.from_pretrained(MID, config=cfg, torch_dtype=torch.float32)
audio_tower = next(m for m in base.modules() if type(m).__name__.endswith("AudioModel"))
del base


feature_extractor = AutoFeatureExtractor.from_pretrained(MID)
TOWER_TAKES_MASK = len(inspect.signature(audio_tower.forward).parameters) > 1

PROJ = {"q_proj", "k_proj", "v_proj", "o_proj"}
targets = [n for n, m in audio_tower.named_modules()
           if isinstance(m, nn.Linear) and PROJ & set(n.split("."))]
audio_tower = get_peft_model(audio_tower, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                                     target_modules=targets))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def run_tower_with_feats(tower, x):                          # -> hidden (B, T', d), features (B, T, n_mels)
    dev = next(tower.parameters()).device
    f = feature_extractor(list(x.detach().cpu().numpy()), sampling_rate=SR, return_tensors="pt")
    feats = f["input_features"].to(dev)
    out = (tower(feats, f["input_features_mask"].to(dev))
           if TOWER_TAKES_MASK else tower(feats))
    return (out[0] if isinstance(out, tuple) else out.last_hidden_state), feats

def run_tower(tower, x):
    return run_tower_with_feats(tower, x)[0]


class WDPModel(nn.Module):
    def __init__(self, tower, num_classes, proj_dim=PROJ_DIM, p_drop=0.1, margin=0.3):
        super().__init__()
        self.encoder = tower
        with torch.no_grad():                                   # measure tower width + feature shape once
            h, feats = run_tower_with_feats(tower, torch.zeros(1, WIN))
        d, n_frames, n_mels = h.shape[-1], feats.shape[1], feats.shape[2]
        self.head_embedding = nn.Linear(d, proj_dim)
        self.decoder        = nn.Sequential(                    # hidden sequence -> whole window's features
            nn.ConvTranspose1d(d, 512, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(128, n_mels, kernel_size=4, stride=2, padding=1),
            nn.AdaptiveAvgPool1d(n_frames)                      # [B, n_mels, n_frames]
        )
        self.classifier = nn.Sequential(nn.Dropout(p_drop), nn.Linear(proj_dim, num_classes))
        self.margin = margin

    def embed(self, x):                                         # (B, WIN) -> (B, 128)
        return self.head_embedding(run_tower(self.encoder, x).mean(1).float())

    def forward(self, x, labels=None, positive=None, negative=None):
        has_triplet = positive is not None and negative is not None
        B = x.shape[0]
        h, feats = run_tower_with_feats(self.encoder, torch.cat([x, positive, negative]) if has_triplet else x)  # one encoder pass
        h = h.float()
        z = self.head_embedding(h.mean(1))
        normed = F.normalize(z, dim=-1)
        anc, logits = normed[:B], self.classifier(z[:B])
        recon = self.decoder(h.transpose(1, 2)).transpose(1, 2)  # (3B, n_frames, n_mels)

        out = {"classifications": logits, "embeddings": anc, "reconstructions": recon[:B]}
        if labels is not None:
            cls_loss = F.cross_entropy(logits, labels)
            rec_loss = F.mse_loss(recon, feats.float())
            out["classification_loss"] = cls_loss
            out["reconstruction_loss"] = rec_loss
            out["total_loss"] = cls_loss + RECON_W * rec_loss
            if has_triplet:
                pos, neg = normed[B:2 * B], normed[2 * B:]
                trip = F.triplet_margin_loss(anc, pos, neg, margin=self.margin)
                out["triplet_loss"] = trip
                out["total_loss"] = out["total_loss"] + 0.5 * trip
        return out

# ---- training: x, positive, negative are (B, WIN); labels (B,) long ----
def train_step(clf, opt, x, labels, positive, negative):
    clf.train()
    fwd = clf(x, labels=labels, positive=positive, negative=negative)
    fwd["total_loss"].backward(); opt.step(); opt.zero_grad()
    acc = (fwd["classifications"].argmax(-1) == labels).float().mean().item()
    return (fwd["total_loss"].item(), fwd["triplet_loss"].item(),
            fwd["classification_loss"].item(), fwd["reconstruction_loss"].item(), acc)

clf = WDPModel(audio_tower, NUM_CLASSES).to(DEVICE)
opt = torch.optim.AdamW([
    {"params": [p for p in clf.encoder.parameters() if p.requires_grad], "lr": 2e-4},
    {"params": clf.head_embedding.parameters(), "lr": 1e-3},
    {"params": clf.classifier.parameters(), "lr": 1e-3},
    {"params": clf.decoder.parameters(), "lr": 1e-3},
])

In [ ]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:



AUDIO  = '/content/drive/MyDrive/dolphin data/l1.wav'
LABELS = '/content/drive/MyDrive/dolphin data/l1.csv'

def load_filtered_waveform(filename):
    waveform, sr = torchaudio.load(filename)
    if waveform.shape[0] > 1:
        # process()'s x.squeeze() only drops size-1 dims, so a multi-channel
        # waveform reaches the Whisper feature extractor as 2D instead of
        # the 1D it expects, breaking its internal padding. Every file this
        # pipeline was built/trained against happens to be mono already, so
        # this is a no-op there -- only multi-channel input hits it.
        waveform = waveform[:1]
    waveform = torchaudio.functional.highpass_biquad(
        waveform, sr, 5000
    )
    waveform = torchaudio.functional.lowpass_biquad(
        waveform, sr, 7900
    )
    return waveform, sr

def instance(label, offset, reverse, by_label, audio):
    negative_label = np.random.choice(list(all_labels.difference({label})))
    x_pos_offset   = np.random.choice(list(by_label[label].difference({offset})))
    x_neg_offset   = np.random.choice(list(by_label[negative_label]))

    y     = reverse[label]
    x     = audio[0, offset:offset + WIN]
    x_pos = audio[0, x_pos_offset:x_pos_offset + WIN]
    x_neg = audio[0, x_neg_offset:x_neg_offset + WIN]
    return x, x_pos, x_neg, y


def batch(instances, i, batch_size=100):
    x   = torch.stack([instances[i][0] for i in range(i, i + batch_size)])
    y   = torch.tensor([instances[i][3] for i in range(i, i + batch_size)])
    pos = torch.stack([instances[i][1] for i in range(i, i + batch_size)])
    neg = torch.stack([instances[i][2] for i in range(i, i + batch_size)])
    return x, pos, neg, y


In [ ]:
LABEL_MAP = {
    2: 'WSTL_DOWN',
    1: 'WSTL_UP',
    4: 'BURST',
    3: 'ECHO',
    0: 'NOISE'
}
reverse    = {v:k for k, v in LABEL_MAP.items()}
all_labels = set(reverse.keys())

In [ ]:
audio      = load_filtered_waveform(AUDIO)[0]
labels     = pd.read_csv(LABELS)
labels     = labels.sample(frac=1).reset_index()
by_label   = labels.groupby(labels.annotation)['offset'].apply(set)

labels['instance'] = labels.apply(lambda x: instance(x['annotation'], x['offset'], reverse, by_label, audio), axis=1)
instances          = list(labels['instance'])

for batch_i in range(0, 25):
    complete = 0
    for i in range(0, len(instances) - 100, 100):
        x, pos, neg, y = batch(instances, i)
        y = y.to(DEVICE)

        total, triplet, classification, recon, acc = train_step(clf, opt, x, y, pos, neg)
        complete += total
        print(f"{total} = {classification} {triplet} {recon} [{acc}]")
    print(f"====================== {batch_i} {complete} ===========================")

In [ ]:
import shutil

OUT = "/content/drive/MyDrive/dolphin/wdp-embeddinggemma2-decoder"
os.makedirs(OUT, exist_ok=True)
clf.encoder.save_pretrained(f"{OUT}/audio-lora")                 # LoRA adapter weights + adapter config
torch.save({"head_embedding": clf.head_embedding.state_dict(),
            "classifier":     clf.classifier.state_dict(),
            "decoder":        clf.decoder.state_dict(),
            "config": {"base_model": MID, "sr": SR, "win": WIN, "num_classes": NUM_CLASSES,
                       "proj_dim": PROJ_DIM, "recon_w": RECON_W, "margin": clf.margin, "label_map": LABEL_MAP}},
           f"{OUT}/wdp-heads.pt")
print(shutil.make_archive(OUT, "zip", OUT))                       # single file to download

# Extend L1 by more noise and L2



In [ ]:
from peft import PeftModel

# ---- reload a saved model (run cell 2 first for MID / feature_extractor / run_tower / WDPModel) ----
OUT = "/content/drive/MyDrive/dolphin/wdp-embeddinggemma2-decoder"
ck  = torch.load(f"{OUT}/wdp-heads.pt", map_location="cpu")

base = AutoModel.from_pretrained(MID, config=cfg, torch_dtype=torch.float32)
audio_tower = next(m for m in base.modules() if type(m).__name__.endswith("AudioModel"))
del base
audio_tower = PeftModel.from_pretrained(audio_tower, f"{OUT}/audio-lora", is_trainable=True)

clf = WDPModel(audio_tower, ck["config"]["num_classes"], ck["config"]["proj_dim"], margin=ck["config"]["margin"])
clf.head_embedding.load_state_dict(ck["head_embedding"])
clf.classifier.load_state_dict(ck["classifier"])
clf.decoder.load_state_dict(ck["decoder"])
clf = clf.to(DEVICE)
opt = torch.optim.AdamW([                                          # fresh optimizer to continue training
    {"params": [p for p in clf.encoder.parameters() if p.requires_grad], "lr": 2e-4},
    {"params": clf.head_embedding.parameters(), "lr": 1e-3},
    {"params": clf.classifier.parameters(), "lr": 1e-3},
    {"params": clf.decoder.parameters(), "lr": 1e-3},
])

In [ ]:
def window_l2(audio, labels):
  with torch.no_grad():
    starts = np.array(labels.starts)
    stops = np.array(labels.stops)
    windowed = []
    labels = []
    for start, stop in zip(starts, stops):
      if stop - start > WIN:
        for i in range(start, stop - WIN, WIN):
          x = audio[:, i:i + WIN]
          fwd = clf(x)
          label = fwd['classifications'].argmax().item()
          if label > 0:
            windowed.append(x)
            labels.append(label)
    return windowed, labels

AUDIO  = '/content/drive/MyDrive/dolphin data/l2.wav'
LABELS = '/content/drive/MyDrive/dolphin data/l2.csv'

audio  = load_filtered_waveform(AUDIO)[0]
labels = pd.read_csv(LABELS)

windows, labels = window_l2(audio, labels)
l1_from_l2      = torch.concat([x[0] for x in windows])

l1_label_names  = [LABEL_MAP[label] for label in labels]
l1_offsets      = [i * WIN for i in range(len(labels))]
l1_csv          = pd.DataFrame({
    'offset': l1_offsets,
    'annotation': l1_label_names})

In [ ]:
l1_offsets[-1] + WIN, l1_from_l2.shape

In [ ]:
OUT = "/content/drive/MyDrive/dolphin data/"
l1_csv.to_csv(f"{OUT}/l1_extended_from_l2.csv", index=False)
torchaudio.save(f"{OUT}/l1_extended_from_l2.wav", l1_from_l2, 44100)


In [ ]:
def window_noise(audio):
  windowed = []
  labels = []
  for i in range(0, len(audio) - WIN, WIN):
    labels.append(0)
    windowed.append(audio[:, i:i + WIN])
  return windowed, labels


AUDIO  = '/content/drive/MyDrive/dolphin data/naive_noise.wav'
audio  = load_filtered_waveform(AUDIO)[0]
windows_noise, labels_noise = window_noise(audio)

all_windows = windows + windows_noise
all_labels = labels + labels_noise

l1_from_l2      = torch.concat([x[0] for x in all_windows])

l1_label_names  = [LABEL_MAP[label] for label in all_labels]
l1_offsets      = [i * WIN for i in range(len(all_labels))]
l1_csv          = pd.DataFrame({
    'offset': l1_offsets,
    'annotation': l1_label_names})

In [ ]:
OUT = "/content/drive/MyDrive/dolphin data/"
l1_csv.to_csv(f"{OUT}/l1_extended_from_l2_noise.csv", index=False)
torchaudio.save(f"{OUT}/l1_extended_from_l2_noise.wav", l1_from_l2, 44100)


# Retrain on mixed L1 and L2

In [ ]:
from peft import PeftModel

# ---- reload a saved model (run cell 2 first for MID / feature_extractor / run_tower / WDPModel) ----
OUT = "/content/drive/MyDrive/dolphin/wdp-embeddinggemma2-decoder"
ck  = torch.load(f"{OUT}/wdp-heads.pt", map_location="cpu")

base = AutoModel.from_pretrained(MID, config=cfg, torch_dtype=torch.float32)
audio_tower = next(m for m in base.modules() if type(m).__name__.endswith("AudioModel"))
del base
audio_tower = PeftModel.from_pretrained(audio_tower, f"{OUT}/audio-lora", is_trainable=True)

clf = WDPModel(audio_tower, ck["config"]["num_classes"], ck["config"]["proj_dim"], margin=ck["config"]["margin"])
clf.head_embedding.load_state_dict(ck["head_embedding"])
clf.classifier.load_state_dict(ck["classifier"])
clf.decoder.load_state_dict(ck["decoder"])
clf = clf.to(DEVICE)
opt = torch.optim.AdamW([                                          # fresh optimizer to continue training
    {"params": [p for p in clf.encoder.parameters() if p.requires_grad], "lr": 2e-4},
    {"params": clf.head_embedding.parameters(), "lr": 1e-3},
    {"params": clf.classifier.parameters(), "lr": 1e-3},
    {"params": clf.decoder.parameters(), "lr": 1e-3},
])

In [ ]:
AUDIO_L1  = '/content/drive/MyDrive/dolphin data/l1.wav'
LABELS_L1 = '/content/drive/MyDrive/dolphin data/l1.csv'
AUDIO_L2  = '/content/drive/MyDrive/dolphin data/l1_extended_from_l2_noise.wav'
LABELS_L2 = '/content/drive/MyDrive/dolphin data/l1_extended_from_l2_noise.csv'


audio_l1  = load_filtered_waveform(AUDIO_L1)[0]
labels_l1 = pd.read_csv(LABELS_L1)
audio_l2  = load_filtered_waveform(AUDIO_L2)[0]
labels_l2 = pd.read_csv(LABELS_L2)

offset = list(labels_l1.offset)[-1] + WIN
labels_l2['offset'] = labels_l2['offset'] + offset
labels = pd.concat([labels_l1, labels_l2])
audio = torch.concat([audio_l1[0], audio_l2[0]])
audio = audio[None, :]

In [ ]:
labels     = labels.sample(frac=1).reset_index()
by_label   = labels.groupby(labels.annotation)['offset'].apply(set)

labels['instance'] = labels.apply(lambda x: instance(x['annotation'], x['offset'], reverse, by_label, audio), axis=1)
instances          = list(labels['instance'])

for batch_i in range(0, 25):
    complete = 0
    for i in range(0, len(instances) - 100, 100):
        x, pos, neg, y = batch(instances, i)
        y = y.to(DEVICE)

        total, triplet, classification, recon, acc = train_step(clf, opt, x, y, pos, neg)
        complete += total
        print(f"{total} = {classification} {triplet} {recon} [{acc}]")
    print(f"====================== {batch_i} {complete} ===========================")

In [ ]:
import shutil

OUT = "/content/drive/MyDrive/dolphin/wdp-embeddinggemma2-decoder_extended"
os.makedirs(OUT, exist_ok=True)
clf.encoder.save_pretrained(f"{OUT}/audio-lora")                 # LoRA adapter weights + adapter config
torch.save({"head_embedding": clf.head_embedding.state_dict(),
            "classifier":     clf.classifier.state_dict(),
            "decoder":        clf.decoder.state_dict(),
            "config": {"base_model": MID, "sr": SR, "win": WIN, "num_classes": NUM_CLASSES,
                       "proj_dim": PROJ_DIM, "recon_w": RECON_W, "margin": clf.margin, "label_map": LABEL_MAP}},
           f"{OUT}/wdp-heads.pt")
print(shutil.make_archive(OUT, "zip", OUT))                       # single file to download